# nb00：環境確認と保存結果の表示

記事の確認セルを収録。**Run allは使わず、設定セルの後、必要な項目だけ実行してください。**
未作成の結果は表示できません。取込み・学習・生成AI送信は行いません。
書込み確認は専用プローブ1個の作成・削除、通信確認はDNS/TCP接続を行います。
表示用Notebookの実行出力には個人用のパスや結果が含まれ得ます。再配布時は出力を除去してください。


In [ ]:
# 自分のVolumeと保存済みの実行IDへ変更して、このセルを実行します。
from pathlib import Path
RAW_ROOT = Path("/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_RAW_VOLUME")
WORK_ROOT = Path("/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_WORK_VOLUME")
SOURCE_2020 = "SET_SOURCE_RUN_ID_2020"
SOURCE_2025 = "SET_SOURCE_RUN_ID_2025"
STRICT_ID = "SET_STRICT_COMPARISON_RUN_ID"
COMPARISON_ID = "SET_PASS_COMPARISON_RUN_ID"
AI_RUN_ID = "SET_PASS_AI_RUN_ID"
REPORT_RUN_ID = "SET_SAVED_REPORT_RUN_ID"

def configured(value):
    if "REPLACE_WITH_" in str(value) or str(value).startswith("SET_"):
        raise ValueError("設定セルのプレースホルダーを自分の値に変更してください: " + str(value))
    return value

def work_path():
    return configured(WORK_ROOT)

def ml_path():
    return work_path() / "ai_runs" / configured(AI_RUN_ID) / "ml"


## Python・Spark・分析ライブラリのバージョン

元記事の確認セル（提供原稿の387行目から）。


In [ ]:
import platform
from importlib.metadata import version

print("Python:", platform.python_version())
print("Spark:", spark.version)
for package in ["numpy", "pandas", "openpyxl", "matplotlib", "scipy", "scikit-learn"]:
    print(package, version(package))


## 6つの原本の存在を確認する

元記事の確認セル（提供原稿の407行目から）。


In [ ]:
from pathlib import Path

RAW = configured(RAW_ROOT)
filenames = ["population_1_1.xlsx", "households_9_1.xlsx", "age_2_7.xlsx"]
missing = []
for year in [2020, 2025]:
    for name in filenames:
        path = RAW / str(year) / name
        exists = path.is_file()
        size = path.stat().st_size if exists else 0
        print(year, name, "exists=", exists, "bytes=", size)
        if not exists or size == 0:
            missing.append(str(path))
if missing:
    raise FileNotFoundError("配置先を確認してください: " + ", ".join(missing))
print("原本6ファイルの存在確認: PASS")


## 出力先への書込みを確認する

元記事の確認セル（提供原稿の434行目から）。
この項目は専用確認ファイル1個を作成・読込み・削除します。


In [ ]:
from pathlib import Path
from uuid import uuid4

WORK = work_path()
if not WORK.is_dir():
    raise FileNotFoundError("出力用Volumeを確認してください: " + str(WORK))
check_dir = WORK / "_checks"
check_dir.mkdir(exist_ok=True)
probe = check_dir / ("write_probe_" + uuid4().hex + ".txt")
with probe.open("x", encoding="utf-8") as stream:
    stream.write("AIDP census write check")
assert probe.read_text(encoding="utf-8") == "AIDP census write check"
probe.unlink()
print("出力用Volumeの読書き確認: PASS")


## 両年の保存結果を別Notebookから確認する

元記事の確認セル（提供原稿の787行目から）。


In [ ]:
import json
from pathlib import Path

WORK = work_path()
run_ids = {
    2020: configured(SOURCE_2020),
    2025: configured(SOURCE_2025),
}
for year, run_id in run_ids.items():
    root = WORK / "runs" / run_id
    meta = json.loads((root / "validation.json").read_text(encoding="utf-8"))
    print(year, meta["status"], meta["rows"], meta["target"])
    assert meta["status"] == "PASS" and meta["census_year"] == year
    assert meta["rows"] == 62
    df = spark.read.parquet(str(root / "curated" / "tokyo_year"))
    df.select("area_code", "area_name", "census_year", "population").orderBy("area_code").show(5, truncate=False)


## 町田市の差分で停止することを確認する

元記事の確認セル（提供原稿の946行目から）。


In [ ]:
import json
from pathlib import Path

root = work_path() / "comparisons" / configured(STRICT_ID)
for filename in ["boundary_comparison.json", "validation.json"]:
    print(filename)
    print(json.dumps(json.loads((root / filename).read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


## 比較テーブルを表示する

元記事の確認セル（提供原稿の1124行目から）。


In [ ]:
import json
from pathlib import Path

root = work_path() / "comparisons" / configured(COMPARISON_ID)
meta = json.loads((root / "validation.json").read_text(encoding="utf-8"))
print({key: meta[key] for key in ["status", "rows", "input_municipality_count", "excluded_area_codes"]})
assert meta["status"] == "PASS" and meta["rows"] == 61
changes = spark.read.parquet(str(root / "curated" / "tokyo_changes"))
changes.select(
    "area_code", "area_name", "population_2020", "population_2025",
    "population_growth_pct", "single_share_change_pp", "elderly_share_change_pp",
    "age_unknown_pct_2020", "age_unknown_pct_2025", "age_unknown_change_pp"
).orderBy("area_code").show(10, truncate=False)


## 学習して評価表を確認する

元記事の確認セル（提供原稿の1243行目から）。


In [ ]:
import json
from pathlib import Path

ml = ml_path()
meta = json.loads((ml / "validation.json").read_text(encoding="utf-8"))
print(json.dumps(meta, ensure_ascii=False, indent=2))
assert meta["status"] == "PASS" and meta["rows"] == 61
spark.read.parquet(str(ml / "k_summary")).orderBy("k").show(truncate=False)
assignments = spark.read.parquet(str(ml / "assignments"))
assignments.groupBy("cluster_id").count().orderBy("cluster_id").show()
assignments.select("area_code", "area_name", "cluster_id").orderBy("area_code").show(15, truncate=False)


## 保存済みの所属と、2地域の数値を表示する

元記事の確認セル（提供原稿の1547行目から）。


In [ ]:
import json
from pathlib import Path


ml_root = ml_path()
meta = json.loads((ml_root / "validation.json").read_text(encoding="utf-8"))
assert meta["status"] == "PASS", "学習結果がPASSではありません"
assert meta["ai_run_id"] == AI_RUN_ID
members = (spark.read.parquet(str(ml_root / "assignments"))
           .select("area_code", "area_name", "cluster_id")
           .orderBy("area_code").collect())
assert len(members) == meta["rows"], "保存された所属の件数が検査記録と異なります"
by_code = {str(r["area_code"]).zfill(5): r.asDict() for r in members}
assert len(by_code) == len(members), "地域コードが重複しています"

def print_table(headers, rows):
    def safe(value):
        return str(value).replace("|", "／").replace("\n", " ")
    print("| " + " | ".join(headers) + " |")
    print("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        print("| " + " | ".join(safe(v) for v in row) + " |")

counts = {}
for row in members:
    cluster_id = int(row["cluster_id"])
    counts[cluster_id] = counts.get(cluster_id, 0) + 1
print("ai_run_id:", AI_RUN_ID)
print("クラスタ別件数:", counts)
cluster0 = [r for r in members if int(r["cluster_id"]) == 0]
print("\nクラスタ0の市区町村一覧")
print_table(["地域コード", "市区町村", "クラスタ"],
            [[r["area_code"], r["area_name"], r["cluster_id"]] for r in cluster0])

# 年別の保存データから、人数・分母・割合を読み解く。
targets = ["13113", "13201"]
years = {}
for year, source_id in [(2020, SOURCE_2020), (2025, SOURCE_2025)]:
    root = work_path() / "runs" / configured(source_id)
    check = json.loads((root / "validation.json").read_text(encoding="utf-8"))
    assert check["status"] == "PASS"
    assert check["census_year"] == year
    data = (spark.read.parquet(str(root / "curated" / "tokyo_year"))
            .filter("area_code IN ('13113', '13201')")
            .select("area_code", "area_name", "population", "age_under15",
                    "age_15_64", "age_65plus", "age_unknown").collect())
    assert len(data) == 2, "2地域の年別データを取得できませんでした"
    years[year] = {str(r["area_code"]).zfill(5): r.asDict() for r in data}
    assert set(years[year]) == set(targets)

detail = []
for code in targets:
    assert code in by_code, f"{code}が分類結果にありません"
    a, b = years[2020][code], years[2025][code]
    known_a = a["age_under15"] + a["age_15_64"] + a["age_65plus"]
    known_b = b["age_under15"] + b["age_15_64"] + b["age_65plus"]
    assert known_a > 0 and known_b > 0
    assert a["age_unknown"] is not None and b["age_unknown"] is not None
    elderly_a, elderly_b = 100*a["age_65plus"]/known_a, 100*b["age_65plus"]/known_b
    unknown_a = 100*a["age_unknown"]/a["population"]
    unknown_b = 100*b["age_unknown"]/b["population"]
    detail.append([
        a["area_name"], by_code[code]["cluster_id"],
        f'{a["age_65plus"]:,} → {b["age_65plus"]:,}',
        f'{known_a:,} → {known_b:,}',
        f'{elderly_a:.2f}% → {elderly_b:.2f}%',
        f'{elderly_b-elderly_a:+.2f}pp',
        f'{unknown_a:.2f}% → {unknown_b:.2f}%',
        f'{unknown_b-unknown_a:+.2f}pp',
    ])
print("\n渋谷区・八王子市の所属と年齢指標（2020年 → 2025年）")
print_table(["市区町村", "クラスタ", "65歳以上人口（人）", "年齢既知人口（人）",
             "高齢者割合", "割合の差", "年齢不詳率", "不詳率の差"], detail)


## 保存されたグラフをNotebookに表示する

元記事の確認セル（提供原稿の1673行目から）。


In [ ]:
from pathlib import Path
from IPython.display import Image, display

charts = ml_path() / "charts"
for name in ["k_selection.png", "feature_changes.png", "share_movements.png"]:
    path = charts / name
    if not path.is_file():
        raise FileNotFoundError(str(path))
    print(name)
    display(Image(filename=str(path)))


## 新宿区と変化が近い市区町村を表示する

元記事の確認セル（提供原稿の1729行目から）。


In [ ]:
path = str(ml_path() / "nearest_neighbors")
spark.read.parquet(path).orderBy("rank").show(truncate=False)


## OCI SDK・cryptographyのバージョン

元記事の確認セル（提供原稿の1850行目から）。


In [ ]:
import oci
import cryptography
print("OCI SDK:", oci.__version__)
print("cryptography:", cryptography.__version__)


## 東京・大阪へのDNS・TCP接続

元記事の確認セル（提供原稿の1870行目から）。
この項目はDNS/TCPのみ確認します。認証やAPI成功を証明しません。


In [ ]:
import socket
for region in ("ap-tokyo-1", "ap-osaka-1"):
    host = f"inference.generativeai.{region}.oci.oraclecloud.com"
    print(f"--- {region} ---")
    try:
        print("DNS:", socket.gethostbyname_ex(host)[2])
        with socket.create_connection((host, 443), timeout=10):
            print("TCP 443: OK")
    except OSError as error:
        print("接続確認:", type(error).__name__, str(error))


## nb03の保存済み検査JSON

元記事の確認セル（提供原稿の1899行目から）。


In [ ]:
import json
from pathlib import Path
ml_root = ml_path()
ml_check = json.loads((ml_root / "validation.json").read_text(encoding="utf-8"))
print({key: ml_check.get(key) for key in ("status", "ai_run_id", "rows", "selected_k")})
assert ml_check["status"] == "PASS"
assert ml_check["rows"] == 61


## 数値の検査結果と説明案を読む

元記事の確認セル（提供原稿の2332行目から）。


In [ ]:
import json
from pathlib import Path

report_dir = work_path() / "ai_runs" / configured(AI_RUN_ID) / "genai" / configured(REPORT_RUN_ID)
print("保存ファイル:")
for path in sorted(report_dir.iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size, "bytes")
validation_path = report_dir / "validation.json"
if validation_path.is_file():
    print(json.dumps(json.loads(validation_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("検査結果は未作成です。nb04の実行結果と送信・応答の保存状態を確認してください。")
draft_path = report_dir / "report_draft.md"
if draft_path.is_file():
    print(draft_path.read_text(encoding="utf-8"))
else:
    print("説明案は未作成です。JSONや根拠値の照合で停止していないか確認してください。")
